# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [3]:
# Task 1 — Multi-series line with highlight
# YOUR CODE HERE
# Task 1 — Multi-Series Line Chart with Highlight

asia_df = df[df['Region'] == 'Asia']

highlight_country = 'China'

fig = go.Figure()

for country in asia_df['Country'].unique():
    
    country_df = asia_df[asia_df['Country'] == country]
    
    if country == highlight_country:
        
        fig.add_trace(
            go.Scatter(
                x=country_df['Year'],
                y=country_df['CO2_Mt'],
                mode='lines',
                line=dict(color='crimson', width=4),
                name=country
            )
        )
        
    else:
        
        fig.add_trace(
            go.Scatter(
                x=country_df['Year'],
                y=country_df['CO2_Mt'],
                mode='lines',
                line=dict(color='#DDDDDD', width=1),
                hoverinfo='skip',
                showlegend=False
            )
        )

# Direct label
latest = asia_df[
    (asia_df['Country'] == highlight_country) &
    (asia_df['Year'] == asia_df['Year'].max())
]

fig.add_annotation(
    x=latest['Year'].values[0],
    y=latest['CO2_Mt'].values[0],
    text='China',
    showarrow=False,
    xshift=30,
    font=dict(size=12, color='crimson')
)

# Layout
fig.update_layout(
    title='China became Asia’s dominant CO2 emitter after rapid industrial growth',
    template='simple_white',
    font=dict(family='Arial', size=12),
    width=1000,
    height=600,
    showlegend=False
)

# Remove clutter
fig.update_xaxes(showgrid=False, title='')
fig.update_yaxes(gridcolor='#EEEEEE', title='CO2 Emissions (Mt)')

fig.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [4]:
# Task 2 — Slopegraph: regional averages
# YOUR CODE HERE
# Task 2 — Slopegraph: Regional Change 2000 vs 2022

# Aggregate regional averages
regional_avg = (
    df.groupby(['Region', 'Year'])['CO2_Mt']
    .mean()
    .reset_index()
)

# Filter only 2000 and 2022
regional_filtered = regional_avg[
    regional_avg['Year'].isin([2000, 2022])
]

# Pivot for slopegraph structure
slope_df = regional_filtered.pivot(
    index='Region',
    columns='Year',
    values='CO2_Mt'
).reset_index()

# Sort by 2022 values
slope_df = slope_df.sort_values(by=2022)

# Create change column
slope_df['Change'] = slope_df[2022] - slope_df[2000]

# Create figure
fig = go.Figure()

for _, row in slope_df.iterrows():
    
    # Colour based on increase/decrease
    color = 'crimson' if row['Change'] > 0 else 'seagreen'
    
    # Add slope line
    fig.add_trace(
        go.Scatter(
            x=[2000, 2022],
            y=[row[2000], row[2022]],
            mode='lines+markers',
            line=dict(color=color, width=3),
            marker=dict(size=8),
            showlegend=False
        )
    )
    
    # Left-side labels
    fig.add_annotation(
        x=2000,
        y=row[2000],
        text=f"{row['Region']} ({row[2000]:.1f})",
        showarrow=False,
        xshift=-70,
        font=dict(size=10)
    )
    
    # Right-side labels
    fig.add_annotation(
        x=2022,
        y=row[2022],
        text=f"{row[2022]:.1f}",
        showarrow=False,
        xshift=35,
        font=dict(size=10)
    )

# Layout styling
fig.update_layout(
    title='Asia increased the most while Europe declined the most',
    template='simple_white',
    font=dict(family='Arial', size=12),
    width=900,
    height=600,
    showlegend=False
)

# Remove clutter
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    zeroline=False,
    title=''
)

fig.update_xaxes(
    tickvals=[2000, 2022],
    showgrid=False,
    title=''
)

fig.show()